# 03 — Download expanded historical GFS forecasts for RDU

This notebook replaces the small temperature-only GFS download with a richer training dataset.

## What changes from `01_download_gfs.ipynb`

Instead of only **5 historical runs**, this notebook downloads **40 historical runs by default**:

- 8 weekly GFS runs per year
- 2021, 2022, 2023, 2024, 2025
- each run initialized at **18Z**
- runs span roughly late July through September 16
- every run uses the same forecast horizon as the assignment:
  - forecast hours `10–120` hourly
  - forecast hours `123–345` every 3 hours

That gives:

`40 runs × 186 forecast points = 7,440 historical forecast rows`

It also re-downloads the final **2026-09-16 18Z** run with the same variables.

## Variables

Core variables downloaded from NOAA GFS:

1. 2 m temperature
2. 2 m dew point
3. 2 m relative humidity
4. 10 m U wind
5. 10 m V wind
6. mean sea-level pressure
7. total cloud cover
8. surface precipitation rate

Useful additional variables:

9. downward shortwave radiation at the surface
10. 850 hPa temperature

The notebook also derives:

- temperature and dew point in °F
- wind speed
- wind direction
- pressure in hPa
- precipitation rate in mm/hour

## Why precipitation rate instead of accumulated precipitation?

GFS accumulated precipitation (`APCP`) can use different accumulation windows at different forecast leads. That makes it easy to accidentally compare unlike quantities.

For the first modeling dataset, this notebook uses **precipitation rate (`PRATE`)** instead. We can add carefully normalized APCP later if validation shows it is useful.

## Important practical note

This is a **large archive job**. GFS is stored as global GRIB fields, so obtaining many historical forecast runs is much heavier than downloading a normal CSV.

This notebook therefore:

- runs a **preflight test** before the bulk job
- saves a **checkpoint after every completed GFS run**
- resumes automatically after interruption
- logs failures instead of silently dropping them
- only writes the final historical CSV after completeness checks pass

Do not start the bulk section if the preflight test fails. Send the preflight output/error back instead.

In [2]:
# Imports
#
# This notebook assumes the same Herbie environment that worked for
# 01_download_gfs.ipynb.
#
# If this import fails, do not change data sources. Send the error back.

from herbie import Herbie

import numpy as np
import pandas as pd
import xarray as xr

from pathlib import Path
import math
import time
import traceback

## 1. Configuration

In [ ]:
RAW_DIR = Path("../../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Approximate RDU airport coordinates used in the original notebook.
RDU_LAT = 35.88
RDU_LON = -78.79

LOCAL_TZ = "America/New_York"

# ---------------------------------------------------------
# Historical training runs
# ---------------------------------------------------------

HISTORICAL_YEARS = [2021, 2022, 2023, 2024, 2025]

# 8 weekly runs per year = 40 historical forecast origins.
#
# They are generated by counting backward in 7-day increments from
# September 16 at 18Z:
# Sep 16, Sep 9, Sep 2, Aug 26, Aug 19, Aug 12, Aug 5, Jul 29.
#
# If 40 runs complete cleanly and we later want even more training data,
# increase this to 10 or 12.
RUNS_PER_YEAR = 8

FINAL_GFS_RUN = pd.Timestamp("2026-09-16 18:00:00", tz="UTC")

# Same forecast horizon as the original notebook and the assignment.
FORECAST_HOURS = (
    list(range(10, 121))
    + list(range(123, 346, 3))
)

# ---------------------------------------------------------
# Output files
# ---------------------------------------------------------

HISTORICAL_OUTPUT = RAW_DIR / "NOAA_GFS_FORECAST_HISTORICAL_EXPANDED.csv"
FUTURE_OUTPUT = RAW_DIR / "NOAA_GFS_FORECAST_FUTURE_EXPANDED.csv"

HISTORICAL_CHECKPOINT = RAW_DIR / "_GFS_HISTORICAL_EXPANDED_CHECKPOINT.csv"
FUTURE_CHECKPOINT = RAW_DIR / "_GFS_FUTURE_EXPANDED_CHECKPOINT.csv"

FAILURE_LOG = RAW_DIR / "_GFS_EXPANDED_FAILURES.csv"

print("Historical years:", HISTORICAL_YEARS)
print("Runs/year:", RUNS_PER_YEAR)
print("Forecast points/run:", len(FORECAST_HOURS))
print(
    "Expected historical rows:",
    len(HISTORICAL_YEARS) * RUNS_PER_YEAR * len(FORECAST_HOURS)
)

Historical years: [2021, 2022, 2023, 2024, 2025]
Runs/year: 8
Forecast points/run: 186
Expected historical rows: 7440


In [4]:
def build_historical_runs(years, runs_per_year):
    """
    Build weekly 18Z GFS initialization times counting backward
    from September 16 of each year.
    """
    runs = []

    for year in years:
        anchor = pd.Timestamp(
            f"{year}-09-16 18:00:00",
            tz="UTC"
        )

        offsets = list(range(runs_per_year - 1, -1, -1))

        for weeks_back in offsets:
            run = anchor - pd.Timedelta(days=7 * weeks_back)
            runs.append(run)

    return runs


HISTORICAL_RUNS = build_historical_runs(
    HISTORICAL_YEARS,
    RUNS_PER_YEAR
)

print("Number of historical runs:", len(HISTORICAL_RUNS))
print("\nFirst 10 runs:")
for x in HISTORICAL_RUNS[:10]:
    print(" ", x)

print("\nLast 10 runs:")
for x in HISTORICAL_RUNS[-10:]:
    print(" ", x)

Number of historical runs: 40

First 10 runs:
  2021-07-29 18:00:00+00:00
  2021-08-05 18:00:00+00:00
  2021-08-12 18:00:00+00:00
  2021-08-19 18:00:00+00:00
  2021-08-26 18:00:00+00:00
  2021-09-02 18:00:00+00:00
  2021-09-09 18:00:00+00:00
  2021-09-16 18:00:00+00:00
  2022-07-29 18:00:00+00:00
  2022-08-05 18:00:00+00:00

Last 10 runs:
  2024-09-09 18:00:00+00:00
  2024-09-16 18:00:00+00:00
  2025-07-29 18:00:00+00:00
  2025-08-05 18:00:00+00:00
  2025-08-12 18:00:00+00:00
  2025-08-19 18:00:00+00:00
  2025-08-26 18:00:00+00:00
  2025-09-02 18:00:00+00:00
  2025-09-09 18:00:00+00:00
  2025-09-16 18:00:00+00:00


## 2. GFS variable search

Herbie subsets GFS GRIB messages by searching the GFS index file.

The combined search below asks for only the fields we need rather than downloading the full GFS file for every forecast hour.

The **preflight cells below are intentionally mandatory**. They verify that the GFS archive and your installed Herbie/cfgrib versions expose the expected fields before the long download begins.

In [5]:
# Core fields that we expect to exist for every forecast point.
SEARCH_PATTERNS = {
    "temp_2m": r":TMP:2 m above",
    "dewpoint_2m": r":DPT:2 m above",
    "rh_2m": r":RH:2 m above",
    "u10": r":UGRD:10 m above",
    "v10": r":VGRD:10 m above",
    "mslp": r":PRMSL:mean sea level",
    "cloud_cover": r":TCDC:entire atmosphere",
    "precip_rate": r":PRATE:surface",

    # Additional useful temperature-prediction features.
    "shortwave": r":DSWRF:surface",
    "temp_850": r":TMP:850 mb",
}

# One combined regex => one Herbie subset operation per forecast hour.
GFS_SEARCH = "(" + "|".join(SEARCH_PATTERNS.values()) + ")"

print(GFS_SEARCH)

(:TMP:2 m above|:DPT:2 m above|:RH:2 m above|:UGRD:10 m above|:VGRD:10 m above|:PRMSL:mean sea level|:TCDC:entire atmosphere|:PRATE:surface|:DSWRF:surface|:TMP:850 mb)


## 3. Mandatory preflight: inspect one archived GFS file

In [6]:
# Use a representative historical GFS file.
PREFLIGHT_INIT = pd.Timestamp("2025-09-16 18:00:00", tz="UTC")
PREFLIGHT_FXX = 24

H_test = Herbie(
    PREFLIGHT_INIT.tz_localize(None),
    model="gfs",
    product="pgrb2.0p25",
    fxx=PREFLIGHT_FXX,
)

print(H_test)

✅ Found ┊ model=gfs ┊ product=pgrb2.0p25 ┊ 2025-Sep-16 18:00 UTC F24 ┊ GRIB2 @ aws ┊ IDX @ local
║HERBIE╠ GFS:pgrb2.0p25


In [7]:
# Check every requested field against the actual GFS inventory.
inventory_results = []

for feature_name, pattern in SEARCH_PATTERNS.items():
    inv = H_test.inventory(pattern)

    count = 0 if inv is None else len(inv)

    inventory_results.append({
        "feature": feature_name,
        "pattern": pattern,
        "inventory_matches": count,
    })

inventory_check = pd.DataFrame(inventory_results)

display(inventory_check)

missing_inventory = inventory_check.loc[
    inventory_check["inventory_matches"] == 0,
    "feature"
].tolist()

if missing_inventory:
    raise RuntimeError(
        "STOP BEFORE BULK DOWNLOAD.\n"
        "The following requested GFS variables were not found in the "
        f"preflight inventory: {missing_inventory}\n\n"
        "Send this table/error back so the search strings can be adjusted. "
        "Do not silently remove those variables."
    )

print("PASS: all requested variable families were found in the inventory.")

,feature,pattern,inventory_matches
0,temp_2m,:TMP:2 m above,1
1,dewpoint_2m,:DPT:2 m above,1
2,rh_2m,:RH:2 m above,1
3,u10,:UGRD:10 m above,1
4,v10,:VGRD:10 m above,1
5,mslp,:PRMSL:mean sea level,1
6,cloud_cover,:TCDC:entire atmosphere,2
7,precip_rate,:PRATE:surface,2
8,shortwave,:DSWRF:surface,1
9,temp_850,:TMP:850 mb,1


PASS: all requested variable families were found in the inventory.


## 4. Extraction helpers

`cfgrib` may return either one xarray Dataset or a list of Datasets because the requested variables live on different vertical levels and use different forecast-step metadata.

The functions below handle both cases and search all returned data variables.

The preflight extraction prints the exact variable names that your local Herbie/cfgrib stack produces.

In [8]:
def as_dataset_list(obj):
    if isinstance(obj, (list, tuple)):
        return list(obj)
    return [obj]


def all_data_variables(ds_or_list):
    """
    Return (variable_name, DataArray) tuples across one Dataset
    or a list of Datasets.
    """
    result = []

    for ds in as_dataset_list(ds_or_list):
        if not isinstance(ds, xr.Dataset):
            continue

        for name in ds.data_vars:
            result.append((name, ds[name]))

    return result


def print_data_variable_diagnostics(ds_or_list):
    rows = []

    for name, da in all_data_variables(ds_or_list):
        rows.append({
            "xarray_name": name,
            "GRIB_shortName": da.attrs.get("GRIB_shortName"),
            "GRIB_name": da.attrs.get("GRIB_name"),
            "GRIB_typeOfLevel": da.attrs.get("GRIB_typeOfLevel"),
            "GRIB_level": da.attrs.get("GRIB_level"),
            "GRIB_units": da.attrs.get("GRIB_units"),
            "GRIB_stepType": da.attrs.get("GRIB_stepType"),
            "GRIB_stepRange": da.attrs.get("GRIB_stepRange"),
        })

    return pd.DataFrame(rows)

In [9]:
# Candidate xarray/cfgrib variable names.
#
# Several aliases are included because names can vary slightly across
# cfgrib/Herbie versions. The preflight diagnostics show what your
# environment actually uses.

VARIABLE_CANDIDATES = {
    "temp_2m": ["t2m", "2t"],
    "dewpoint_2m": ["d2m", "2d"],
    "rh_2m": ["r2", "2r", "rh2m"],
    "u10": ["u10", "10u"],
    "v10": ["v10", "10v"],
    "mslp": ["prmsl", "msl", "mslet"],
    "cloud_cover": ["tcc"],
    "precip_rate": ["prate"],
    "shortwave": ["dswrf", "ssrd"],
    "temp_850": ["t", "tmp"],
}


def find_variable(ds_or_list, candidate_names):
    candidate_names = {str(x).lower() for x in candidate_names}

    # First preference: xarray variable name.
    for name, da in all_data_variables(ds_or_list):
        if str(name).lower() in candidate_names:
            return da

    # Fallback: GRIB short name / CF variable name.
    for name, da in all_data_variables(ds_or_list):
        attrs = da.attrs

        possible = {
            str(attrs.get("GRIB_shortName", "")).lower(),
            str(attrs.get("GRIB_cfVarName", "")).lower(),
        }

        if possible & candidate_names:
            return da

    return None


def choose_rdu_point(da):
    """
    Select the nearest GFS grid point to RDU and return:
    value, grid_lat, grid_lon
    """
    lat_name = None
    lon_name = None

    for candidate in ["latitude", "lat"]:
        if candidate in da.coords:
            lat_name = candidate
            break

    for candidate in ["longitude", "lon"]:
        if candidate in da.coords:
            lon_name = candidate
            break

    if lat_name is None or lon_name is None:
        raise KeyError(
            f"Could not identify latitude/longitude coordinates for {da.name}. "
            f"Coords: {list(da.coords)}"
        )

    lon_values = da[lon_name]

    target_lon = RDU_LON
    if float(lon_values.max()) > 180:
        target_lon = RDU_LON % 360

    point = da.sel(
        {
            lat_name: RDU_LAT,
            lon_name: target_lon,
        },
        method="nearest"
    ).squeeze(drop=True)

    values = np.asarray(point.values)

    if values.size != 1:
        raise ValueError(
            f"Expected one value for {da.name} at RDU, "
            f"but got shape {values.shape}."
        )

    value = float(values.reshape(-1)[0])

    grid_lat = float(
        np.asarray(point[lat_name].values).reshape(-1)[0]
    )
    grid_lon = float(
        np.asarray(point[lon_name].values).reshape(-1)[0]
    )

    return value, grid_lat, grid_lon

In [10]:
def get_value(ds_or_list, feature_name, required=True):
    da = find_variable(
        ds_or_list,
        VARIABLE_CANDIDATES[feature_name]
    )

    if da is None:
        if required:
            available = [name for name, _ in all_data_variables(ds_or_list)]
            raise KeyError(
                f"Could not find extracted variable for '{feature_name}'. "
                f"Available xarray variables: {available}"
            )
        return np.nan, np.nan, np.nan, {}

    value, grid_lat, grid_lon = choose_rdu_point(da)

    metadata = {
        "grib_short_name": da.attrs.get("GRIB_shortName"),
        "grib_name": da.attrs.get("GRIB_name"),
        "grib_units": da.attrs.get("GRIB_units"),
        "grib_step_type": da.attrs.get("GRIB_stepType"),
        "grib_step_range": da.attrs.get("GRIB_stepRange"),
    }

    return value, grid_lat, grid_lon, metadata


def kelvin_to_c(k):
    return k - 273.15


def c_to_f(c):
    return c * 9 / 5 + 32


def wind_speed(u, v):
    return math.sqrt(u**2 + v**2)


def wind_direction_from(u, v):
    """
    Meteorological wind direction: direction FROM which wind blows.
    0/360 = north, 90 = east, 180 = south, 270 = west.
    """
    return (270 - math.degrees(math.atan2(v, u))) % 360

In [11]:
# Actually download and decode one combined subset.
#
# If this cell fails, STOP and send the full error/output back.

preflight_ds = H_test.xarray(
    GFS_SEARCH,
    remove_grib=True
)

diagnostics = print_data_variable_diagnostics(preflight_ds)
display(diagnostics)

/home/konurnordberg/.pyenv/versions/3.10.21/lib/python3.10/site-packages/herbie/core.py:978: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  logic = df.search_this.str.contains(search)


Note: Returning a list of [8] xarray.Datasets because cfgrib opened with multiple hypercubes.


,xarray_name,GRIB_shortName,GRIB_name,GRIB_typeOfLevel,GRIB_level,GRIB_units,GRIB_stepType,GRIB_stepRange
0,tcc,tcc,Total Cloud Cover,atmosphere,None,%,instant,24
1,tcc,tcc,Total Cloud Cover,atmosphere,None,%,avg,18-24
2,u10,10u,10 metre U wind component,heightAboveGround,None,m s**-1,instant,24
3,v10,10v,10 metre V wind component,heightAboveGround,None,m s**-1,instant,24
4,t2m,2t,2 metre temperature,heightAboveGround,None,K,instant,24
5,d2m,2d,2 metre dewpoint temperature,heightAboveGround,None,K,instant,24
6,r2,2r,2 metre relative humidity,heightAboveGround,None,%,instant,24
7,t,t,Temperature,isobaricInhPa,None,K,instant,24
8,prmsl,prmsl,Pressure reduced to MSL,meanSea,None,Pa,instant,24
9,prate,prate,Precipitation rate,surface,None,kg m**-2 s**-1,instant,24


In [12]:
# Verify that all required modeling variables can be extracted.

CORE_FEATURES = [
    "temp_2m",
    "dewpoint_2m",
    "rh_2m",
    "u10",
    "v10",
    "mslp",
    "cloud_cover",
    "precip_rate",
]

OPTIONAL_FEATURES = [
    "shortwave",
    "temp_850",
]

preflight_values = {}

for feature in CORE_FEATURES:
    value, lat, lon, meta = get_value(
        preflight_ds,
        feature,
        required=True
    )
    preflight_values[feature] = value

for feature in OPTIONAL_FEATURES:
    value, lat, lon, meta = get_value(
        preflight_ds,
        feature,
        required=False
    )
    preflight_values[feature] = value

display(
    pd.DataFrame(
        {
            "feature": list(preflight_values.keys()),
            "value": list(preflight_values.values()),
        }
    )
)

if any(pd.isna(preflight_values[x]) for x in CORE_FEATURES):
    raise RuntimeError(
        "STOP: at least one core variable is missing in the extraction test."
    )

print("PASS: combined GFS subset downloaded and all core variables extracted.")
print("It is now safe to run the bulk-download section.")

,feature,value
0,temp_2m,293.244385
1,dewpoint_2m,288.122192
2,rh_2m,73.000000
3,u10,2.920576
4,v10,-0.874055
5,mslp,101653.898438
6,cloud_cover,96.500000
7,precip_rate,0.000000
8,shortwave,NaN
9,temp_850,284.565125


PASS: combined GFS subset downloaded and all core variables extracted.
It is now safe to run the bulk-download section.


## 5. Function to download one RDU forecast point

In [13]:
def get_gfs_point(init_time, forecast_hour):
    """
    Download selected GFS variables for one run/forecast hour
    and extract the nearest 0.25-degree GFS grid point to RDU.
    """
    init_time = pd.Timestamp(init_time)

    if init_time.tzinfo is None:
        init_time = init_time.tz_localize("UTC")
    else:
        init_time = init_time.tz_convert("UTC")

    H = Herbie(
        init_time.tz_localize(None),
        model="gfs",
        product="pgrb2.0p25",
        fxx=int(forecast_hour),
        verbose=False,
    )

    ds = H.xarray(
        GFS_SEARCH,
        remove_grib=True,
    )

    # Core raw variables
    temp_k, grid_lat, grid_lon, _ = get_value(
        ds, "temp_2m", required=True
    )
    dew_k, _, _, _ = get_value(
        ds, "dewpoint_2m", required=True
    )
    rh_pct, _, _, _ = get_value(
        ds, "rh_2m", required=True
    )
    u10, _, _, _ = get_value(
        ds, "u10", required=True
    )
    v10, _, _, _ = get_value(
        ds, "v10", required=True
    )
    mslp_pa, _, _, _ = get_value(
        ds, "mslp", required=True
    )
    tcc_pct, _, _, _ = get_value(
        ds, "cloud_cover", required=True
    )
    prate, _, _, prate_meta = get_value(
        ds, "precip_rate", required=True
    )

    # Additional useful variables
    shortwave, _, _, shortwave_meta = get_value(
        ds, "shortwave", required=False
    )
    temp_850_k, _, _, _ = get_value(
        ds, "temp_850", required=False
    )

    # Unit conversions
    temp_c = kelvin_to_c(temp_k)
    temp_f = c_to_f(temp_c)

    dew_c = kelvin_to_c(dew_k)
    dew_f = c_to_f(dew_c)

    wind_mps = wind_speed(u10, v10)
    wind_mph = wind_mps * 2.2369362920544
    wind_dir = wind_direction_from(u10, v10)

    mslp_hpa = mslp_pa / 100.0

    # GFS PRATE is kg m^-2 s^-1.
    # 1 kg m^-2 liquid water = 1 mm.
    precip_rate_mm_hr = prate * 3600.0

    if pd.isna(temp_850_k):
        temp_850_c = np.nan
        temp_850_f = np.nan
    else:
        temp_850_c = kelvin_to_c(temp_850_k)
        temp_850_f = c_to_f(temp_850_c)

    valid_time_utc = init_time + pd.Timedelta(
        hours=int(forecast_hour)
    )
    valid_time_local = valid_time_utc.tz_convert(LOCAL_TZ)

    # Store grid longitude in both native GFS convention and -180..180.
    grid_lon_normalized = (
        ((grid_lon + 180) % 360) - 180
    )

    return {
        "gfs_init_utc": init_time.isoformat(),
        "forecast_hour": int(forecast_hour),
        "valid_time_utc": valid_time_utc.isoformat(),
        "valid_time_local": valid_time_local.isoformat(),

        "gfs_temp_k": temp_k,
        "gfs_temp_c": temp_c,
        "gfs_temp_f": temp_f,

        "gfs_dewpoint_k": dew_k,
        "gfs_dewpoint_c": dew_c,
        "gfs_dewpoint_f": dew_f,

        "gfs_relative_humidity_pct": rh_pct,

        "gfs_u10_mps": u10,
        "gfs_v10_mps": v10,
        "gfs_wind_speed_mps": wind_mps,
        "gfs_wind_speed_mph": wind_mph,
        "gfs_wind_direction_deg": wind_dir,

        "gfs_mslp_pa": mslp_pa,
        "gfs_mslp_hpa": mslp_hpa,

        "gfs_total_cloud_cover_pct": tcc_pct,

        "gfs_precip_rate_kg_m2_s": prate,
        "gfs_precip_rate_mm_hr": precip_rate_mm_hr,
        "gfs_precip_step_type": prate_meta.get("grib_step_type"),
        "gfs_precip_step_range": prate_meta.get("grib_step_range"),

        "gfs_downward_shortwave_w_m2": shortwave,
        "gfs_shortwave_step_type": shortwave_meta.get("grib_step_type"),
        "gfs_shortwave_step_range": shortwave_meta.get("grib_step_range"),

        "gfs_temp_850_k": temp_850_k,
        "gfs_temp_850_c": temp_850_c,
        "gfs_temp_850_f": temp_850_f,

        "gfs_grid_lat": grid_lat,
        "gfs_grid_lon_native": grid_lon,
        "gfs_grid_lon": grid_lon_normalized,
    }

In [14]:
# One complete row as a final smoke test.
#
# This should return a dictionary containing all core variables.

test_row = get_gfs_point(
    PREFLIGHT_INIT,
    PREFLIGHT_FXX
)

pd.Series(test_row)

/home/konurnordberg/.pyenv/versions/3.10.21/lib/python3.10/site-packages/herbie/core.py:978: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  logic = df.search_this.str.contains(search)


gfs_init_utc                   2025-09-16T18:00:00+00:00
forecast_hour                                         24
valid_time_utc                 2025-09-17T18:00:00+00:00
valid_time_local               2025-09-17T14:00:00-04:00
gfs_temp_k                                    293.244385
gfs_temp_c                                     20.094385
gfs_temp_f                                     68.169893
gfs_dewpoint_k                                288.122192
gfs_dewpoint_c                                 14.972192
gfs_dewpoint_f                                 58.949946
gfs_relative_humidity_pct                           73.0
gfs_u10_mps                                     2.920576
gfs_v10_mps                                    -0.874055
gfs_wind_speed_mps                              3.048563
gfs_wind_speed_mph                              6.819441
gfs_wind_direction_deg                        286.661122
gfs_mslp_pa                                101653.898438
gfs_mslp_hpa                   

## 6. Checkpoint / resume helpers

The long download should be restartable.

A row is uniquely identified by:

`(gfs_init_utc, forecast_hour)`

If the notebook is interrupted, rerun from the top. Completed rows in the checkpoint file will be skipped automatically.

In [15]:
KEY_COLUMNS = [
    "gfs_init_utc",
    "forecast_hour",
]


def normalize_init_key(x):
    ts = pd.Timestamp(x)

    if ts.tzinfo is None:
        ts = ts.tz_localize("UTC")
    else:
        ts = ts.tz_convert("UTC")

    return ts.isoformat()


def load_checkpoint(path):
    if not path.exists():
        return pd.DataFrame()

    df = pd.read_csv(path)

    if len(df) == 0:
        return df

    df["gfs_init_utc"] = df["gfs_init_utc"].map(
        normalize_init_key
    )
    df["forecast_hour"] = pd.to_numeric(
        df["forecast_hour"],
        errors="raise"
    ).astype(int)

    return df


def completed_keys(df):
    if len(df) == 0:
        return set()

    return set(
        zip(
            df["gfs_init_utc"],
            df["forecast_hour"],
        )
    )


def save_checkpoint(df, path):
    df = (
        df
        .drop_duplicates(
            subset=KEY_COLUMNS,
            keep="last"
        )
        .sort_values(KEY_COLUMNS)
        .reset_index(drop=True)
    )

    df.to_csv(path, index=False)

    return df


def append_failure(failure):
    failure_df = pd.DataFrame([failure])

    if FAILURE_LOG.exists():
        old = pd.read_csv(FAILURE_LOG)
        failure_df = pd.concat(
            [old, failure_df],
            ignore_index=True
        )

    failure_df.to_csv(
        FAILURE_LOG,
        index=False
    )

## 7. Bulk downloader

This downloads **one GFS run at a time** and checkpoints after the run finishes.

`MAX_RUNS_THIS_EXECUTION` is normally `None`, meaning process all unfinished runs.

If you want to test the mechanism before committing to the full job, temporarily set it to `1`. After that run succeeds, set it back to `None` and rerun the cell.

In [16]:
MAX_RUNS_THIS_EXECUTION = None

# Tiny delay between forecast files.
# This is intentionally conservative rather than hammering the archive.
SLEEP_BETWEEN_POINTS_SECONDS = 0.05

In [17]:
def download_runs(
    init_times,
    forecast_hours,
    checkpoint_path,
    max_runs_this_execution=None,
):
    existing = load_checkpoint(checkpoint_path)
    done = completed_keys(existing)

    rows = existing.to_dict("records")

    runs_processed_now = 0

    for run_number, init_time in enumerate(init_times, start=1):
        init_key = normalize_init_key(init_time)

        missing_for_run = [
            int(fxx)
            for fxx in forecast_hours
            if (init_key, int(fxx)) not in done
        ]

        if not missing_for_run:
            print(
                f"[{run_number}/{len(init_times)}] "
                f"{init_key}: already complete"
            )
            continue

        if (
            max_runs_this_execution is not None
            and runs_processed_now >= max_runs_this_execution
        ):
            print(
                "\nReached MAX_RUNS_THIS_EXECUTION. "
                "Checkpoint is safe."
            )
            break

        print(
            f"\n[{run_number}/{len(init_times)}] "
            f"Processing {init_key}"
        )
        print(
            f"Missing forecast points for this run: "
            f"{len(missing_for_run)}"
        )

        run_failed = False

        for point_number, forecast_hour in enumerate(
            missing_for_run,
            start=1
        ):
            print(
                f"\r  F{forecast_hour:03d} "
                f"({point_number}/{len(missing_for_run)})",
                end="",
                flush=True
            )

            try:
                row = get_gfs_point(
                    init_time,
                    forecast_hour
                )

                rows.append(row)
                done.add(
                    (
                        row["gfs_init_utc"],
                        row["forecast_hour"],
                    )
                )

            except Exception as e:
                print(
                    f"\nFAILED: {init_key}, "
                    f"F{forecast_hour:03d}"
                )
                print(repr(e))

                append_failure({
                    "gfs_init_utc": init_key,
                    "forecast_hour": forecast_hour,
                    "error_type": type(e).__name__,
                    "error": str(e),
                    "traceback_tail": traceback.format_exc()[-2000:],
                })

                run_failed = True

            time.sleep(
                SLEEP_BETWEEN_POINTS_SECONDS
            )

        # Save everything collected so far after each run.
        current = pd.DataFrame(rows)
        current = save_checkpoint(
            current,
            checkpoint_path
        )

        rows = current.to_dict("records")
        done = completed_keys(current)

        print(
            f"\nCheckpoint saved: {checkpoint_path}"
        )
        print(
            f"Total completed rows: {len(current):,}"
        )

        if run_failed:
            print(
                "This run had at least one failure. "
                "Rerunning later will retry the missing points."
            )

        runs_processed_now += 1

    return load_checkpoint(checkpoint_path)

## 8. Download the 40 historical runs

In [18]:
historical_gfs = download_runs(
    init_times=HISTORICAL_RUNS,
    forecast_hours=FORECAST_HOURS,
    checkpoint_path=HISTORICAL_CHECKPOINT,
    max_runs_this_execution=MAX_RUNS_THIS_EXECUTION,
)

print("\nHistorical checkpoint shape:", historical_gfs.shape)
display(historical_gfs.head())
display(historical_gfs.tail())


[1/40] Processing 2021-07-29T18:00:00+00:00
Missing forecast points for this run: 186
  F010 (1/186)

/home/konurnordberg/.pyenv/versions/3.10.21/lib/python3.10/site-packages/herbie/core.py:978: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  logic = df.search_this.str.contains(search)


  F011 (2/186)

/home/konurnordberg/.pyenv/versions/3.10.21/lib/python3.10/site-packages/herbie/core.py:978: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  logic = df.search_this.str.contains(search)


  F012 (3/186)

...

  F058 (49/186)

/home/konurnordberg/.pyenv/versions/3.10.21/lib/python3.10/site-packages/herbie/core.py:978: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  logic = df.search_this.str.contains(search)


  F059 (50/186)

/home/konurnordberg/.pyenv/versions/3.10.21/lib/python3.10/site-packages/herbie/core.py:978: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  logic = df.search_this.str.contains(search)


KeyboardInterrupt: 

## 9. Historical completeness and quality checks

In [ ]:
expected_historical_keys = {
    (
        normalize_init_key(init_time),
        int(forecast_hour),
    )
    for init_time in HISTORICAL_RUNS
    for forecast_hour in FORECAST_HOURS
}

actual_historical_keys = completed_keys(
    historical_gfs
)

missing_historical_keys = sorted(
    expected_historical_keys
    - actual_historical_keys
)

unexpected_historical_keys = sorted(
    actual_historical_keys
    - expected_historical_keys
)

print(
    "Expected historical rows:",
    f"{len(expected_historical_keys):,}"
)
print(
    "Actual historical rows:",
    f"{len(actual_historical_keys):,}"
)
print(
    "Missing rows:",
    f"{len(missing_historical_keys):,}"
)
print(
    "Unexpected rows:",
    f"{len(unexpected_historical_keys):,}"
)

if missing_historical_keys:
    print("\nFirst 20 missing keys:")
    for key in missing_historical_keys[:20]:
        print(key)

    raise RuntimeError(
        "Historical GFS download is not complete yet. "
        "Rerun the bulk download cell to retry missing points. "
        "Do not promote the checkpoint to the final dataset yet."
    )

if unexpected_historical_keys:
    raise RuntimeError(
        "Checkpoint contains unexpected run/hour combinations. "
        "Inspect before continuing."
    )

print("\nPASS: all expected historical GFS rows are present.")

In [ ]:
CORE_OUTPUT_COLUMNS = [
    "gfs_temp_f",
    "gfs_dewpoint_f",
    "gfs_relative_humidity_pct",
    "gfs_u10_mps",
    "gfs_v10_mps",
    "gfs_mslp_hpa",
    "gfs_total_cloud_cover_pct",
    "gfs_precip_rate_mm_hr",
]

print("Core-variable missingness:")
core_missingness = (
    historical_gfs[CORE_OUTPUT_COLUMNS]
    .isna()
    .mean()
    .mul(100)
    .round(4)
    .sort_values(ascending=False)
)

display(
    core_missingness.to_frame("missing_pct")
)

if (core_missingness > 0).any():
    raise RuntimeError(
        "At least one core GFS field contains missing values. "
        "Inspect before writing the final historical CSV."
    )

optional_cols = [
    "gfs_downward_shortwave_w_m2",
    "gfs_temp_850_f",
]

print("\nOptional-variable missingness:")
display(
    historical_gfs[optional_cols]
    .isna()
    .mean()
    .mul(100)
    .round(4)
    .sort_values(ascending=False)
    .to_frame("missing_pct")
)

print("\nBasic range checks:")
print(
    historical_gfs[
        [
            "gfs_temp_f",
            "gfs_dewpoint_f",
            "gfs_relative_humidity_pct",
            "gfs_wind_speed_mph",
            "gfs_mslp_hpa",
            "gfs_total_cloud_cover_pct",
            "gfs_precip_rate_mm_hr",
        ]
    ].describe().T
)

In [ ]:
# Promote the complete checkpoint to the final historical raw-data file.

historical_final = (
    historical_gfs
    .drop_duplicates(
        subset=KEY_COLUMNS,
        keep="last"
    )
    .sort_values(KEY_COLUMNS)
    .reset_index(drop=True)
)

historical_final.to_csv(
    HISTORICAL_OUTPUT,
    index=False
)

print("Saved:", HISTORICAL_OUTPUT)
print("Rows:", f"{len(historical_final):,}")
print("Runs:", historical_final["gfs_init_utc"].nunique())
print("Columns:", len(historical_final.columns))

## 10. Re-download the final 2026 GFS run with the same variables

This uses the same feature extraction as the historical runs.

That is important: **the model must see the same kinds of predictor variables during training and during the final assignment forecast.**

In [ ]:
future_gfs = download_runs(
    init_times=[FINAL_GFS_RUN],
    forecast_hours=FORECAST_HOURS,
    checkpoint_path=FUTURE_CHECKPOINT,
    max_runs_this_execution=None,
)

print("\nFuture checkpoint shape:", future_gfs.shape)
display(future_gfs.head())
display(future_gfs.tail())

In [ ]:
expected_future_keys = {
    (
        normalize_init_key(FINAL_GFS_RUN),
        int(forecast_hour),
    )
    for forecast_hour in FORECAST_HOURS
}

actual_future_keys = completed_keys(
    future_gfs
)

missing_future_keys = sorted(
    expected_future_keys
    - actual_future_keys
)

print("Expected future rows:", len(expected_future_keys))
print("Actual future rows:", len(actual_future_keys))
print("Missing future rows:", len(missing_future_keys))

if missing_future_keys:
    print("\nMissing:")
    for key in missing_future_keys[:20]:
        print(key)

    raise RuntimeError(
        "Final 2026 GFS run is incomplete. "
        "Rerun the download cell to retry missing points."
    )

future_core_missingness = (
    future_gfs[CORE_OUTPUT_COLUMNS]
    .isna()
    .mean()
)

if (future_core_missingness > 0).any():
    display(
        future_core_missingness
        .mul(100)
        .to_frame("missing_pct")
    )

    raise RuntimeError(
        "At least one core variable is missing in the final 2026 GFS run."
    )

print("PASS: final 2026 GFS run is complete.")

In [ ]:
future_final = (
    future_gfs
    .drop_duplicates(
        subset=KEY_COLUMNS,
        keep="last"
    )
    .sort_values(KEY_COLUMNS)
    .reset_index(drop=True)
)

future_final.to_csv(
    FUTURE_OUTPUT,
    index=False
)

print("Saved:", FUTURE_OUTPUT)
print("Rows:", len(future_final))
print("Columns:", len(future_final.columns))

## 11. Final audit

At the end of a successful run, the two important new files are:

- `../data/raw/NOAA_GFS_FORECAST_HISTORICAL_EXPANDED.csv`
- `../data/raw/NOAA_GFS_FORECAST_FUTURE_EXPANDED.csv`

The old temperature-only files are intentionally **not overwritten**. Keep them until we have inspected the expanded files together.

The checkpoint files can also remain in `data/raw/` for now. Once the final datasets are verified and committed/backed up, we can decide whether to delete or `.gitignore` them.

In [ ]:
# Reload from disk and summarize.

hist_check = pd.read_csv(
    HISTORICAL_OUTPUT
)

future_check = pd.read_csv(
    FUTURE_OUTPUT
)

summary = pd.DataFrame(
    [
        {
            "dataset": HISTORICAL_OUTPUT.name,
            "rows": len(hist_check),
            "runs": hist_check["gfs_init_utc"].nunique(),
            "columns": len(hist_check.columns),
            "min_valid_time": hist_check["valid_time_utc"].min(),
            "max_valid_time": hist_check["valid_time_utc"].max(),
        },
        {
            "dataset": FUTURE_OUTPUT.name,
            "rows": len(future_check),
            "runs": future_check["gfs_init_utc"].nunique(),
            "columns": len(future_check.columns),
            "min_valid_time": future_check["valid_time_utc"].min(),
            "max_valid_time": future_check["valid_time_utc"].max(),
        },
    ]
)

display(summary)

print("\nHistorical columns:")
print(hist_check.columns.tolist())